# One neuron, several inputs

So far, we have trained one number at a time. In this notebook, one prediction will use several input numbers. The goal is to keep the math visible before we let any helper object do automatic gradient bookkeeping.

## Dot product: multiply matching things, then add them

A tiny neuron gives each input one matching weight. The dot product means: multiply each input by its matching weight, then add those products together. After that, add the bias.

```text
prediction = w0*x0 + w1*x1 + bias
```

## Shape table for this first example

Shape means how many numbers each thing contains. The inputs and weights both have two numbers because each input needs one matching weight.

| Thing | Value | Shape meaning |
|---|---:|---|
| `inputs` | `[4.0, 2.0]` | 2 numbers |
| `weights` | `[0.25, 3.0]` | 2 numbers |
| `bias` | `-1.0` | 1 number |
| `prediction` | one number | scalar output |

## Hand gradients for the prediction

For the prediction formula only, each weight's gradient is its matching input:

```text
d_prediction/d_weight_i = input_i
```

The bias is added directly, so its prediction gradient is always `1`:

```text
d_prediction/d_bias = 1
```

In [1]:
inputs = [4.0, 2.0]
weights = [0.25, 3.0]
bias = -1.0

product_0 = weights[0] * inputs[0]
product_1 = weights[1] * inputs[1]

prediction = product_0 + product_1 + bias

weight_grads_for_prediction = [inputs[0], inputs[1]]
bias_grad_for_prediction = 1.0

print("product_0:", product_0)
print("product_1:", product_1)
print("prediction:", prediction)
print("weight grads for prediction:", weight_grads_for_prediction)
print("bias grad for prediction:", bias_grad_for_prediction)

product_0: 1.0
product_1: 6.0
prediction: 6.0
weight grads for prediction: [4.0, 2.0]
bias grad for prediction: 1.0


## Reading the output

The two products are the two matched input-weight pairs. The prediction is the sum of those products plus the bias. The weight-gradient list has the same length as `weights` because each weight gets one matching gradient.

## Checkpoint: which input controls which weight gradient?

For this prediction:

```text
prediction = w0*x0 + w1*x1 + bias
```

the gradient for each weight is the input at the same position.

Using:

```python
inputs = [4.0, 2.0]
weights = [0.25, 3.0]
```

answer these before moving on:

1. Which input controls `d_prediction/d_weight_0`?
2. Which input controls `d_prediction/d_weight_1`?
3. Why is `d_prediction/d_bias` equal to `1`?

## Rebuild the prediction with the shared `Value` helper

Now use the `Value` helper from `src/tiny_mlp_course/value.py`. This keeps the same prediction formula, but each number can also store a gradient. Calling `prediction.backward()` asks: how does the prediction change when each earlier value changes?

In [2]:
from tiny_mlp_course.value import Value

x0 = Value(4.0)
x1 = Value(2.0)

w0 = Value(0.25)
w1 = Value(3.0)

bias = Value(-1.0)

prediction = (w0 * x0) + (w1 * x1) + bias

prediction.backward()

print("prediction:", prediction)
print("weight grads for prediction:", w0.grad, w1.grad, end="")
print("bias grad for prediction:", bias.grad)

prediction: Value(data=6.0, grad=1.0)
weight grads for prediction: 4.0 2.0bias grad for prediction: 1.0


## Compare helper gradients to the hand gradients

The helper-computed gradients should match the hand rules from above. For the prediction only, `w0.grad` should be `x0`, `w1.grad` should be `x1`, and `bias.grad` should be `1`. This is the check that the shared helper is doing the same bookkeeping we did by hand.

## Add a target and squared-error loss

The prediction gradient told us how the prediction changes when a weight changes. For training, we care about the loss. The target is the value we wanted, the error is `prediction - target`, and the squared-error loss is `error ** 2`.

The shared `Value` helper supports subtraction, so we can write `prediction - target` directly. The result is still a `Value`, which means the error step stays connected for `.backward()`.

In [3]:
from tiny_mlp_course.value import Value

x0 = Value(4.0)
x1 = Value(2.0)

w0 = Value(0.25)
w1 = Value(3.0)

bias = Value(-1.0)
target = Value(10.0)

prediction = (w0 * x0) + (w1 * x1) + bias
error = prediction - target
loss = error**2

loss.backward()

print("prediction:", prediction.data)
print("target:", target.data)
print("error:", error.data)
print("loss:", loss.data)
print("w0 grad:", w0.grad)
print("w1 grad:", w1.grad)
print("bias grad:", bias.grad)

prediction: 6.0
target: 10.0
error: -4.0
loss: 16.0
w0 grad: -32.0
w1 grad: -16.0
bias grad: -8.0


## Reading the loss-gradient output

These gradients are now loss gradients, not prediction gradients. The prediction is `6.0`, the target is `10.0`, and the error is `-4.0`, so the loss is `16.0`. Because the loss includes the error step, each weight gradient is the matching input multiplied by the loss's sensitivity to the prediction.

## Gradient notation for the loss output

Use `d..._d...` names to show how much the first thing changes when the second thing changes. For example, `dloss_dw1` means how much the loss changes when `w1` changes.

For this example:

```text
pred = w0*x0 + w1*x1 + b
error = pred - target
loss = error**2
```

Prediction gradients:

```text
dpred_dw0 = x0 = 4.0
dpred_dw1 = x1 = 2.0
dpred_db = 1.0
```

First the loss flows back through the error:

```text
dloss_derror = 2 * error
             = 2 * -4.0
             = -8.0

derror_dpred = 1.0

dloss_dpred = dloss_derror * derror_dpred
            = -8.0 * 1.0
            = -8.0
```

Then the loss flows from the prediction to each parameter:

```text
dloss_dw0 = dloss_dpred * dpred_dw0
          = -8.0 * 4.0
          = -32.0

dloss_dw1 = dloss_dpred * dpred_dw1
          = -8.0 * 2.0
          = -16.0

dloss_db = dloss_dpred * dpred_db
         = -8.0 * 1.0
         = -8.0
```

The printed gradients are these same values: `w0.grad = dloss_dw0`, `w1.grad = dloss_dw1`, and `bias.grad = dloss_db`.

## Put the `Value` neuron prediction in a reusable function

The earlier `Value` example wrote `(w0 * x0) + (w1 * x1) + bias` by hand. This function keeps the same graph-building operations, but it loops over the input-weight pairs so the code can handle any matching list length.

The important detail is that `weighted_sum` starts as `Value(0.0)` and each step uses `input_value * weight`. That keeps the operation history connected, so calling `.backward()` on the prediction can still fill in each weight's gradient.

In [23]:
from tiny_mlp_course.value import Value


def predict_one_neuron(
    inputs: list[Value],
    weights: list[Value],
    bias: Value,
) -> Value:
    if len(inputs) != len(weights):
        raise ValueError("inputs and weights must have the same length")

    weighted_sum = Value(0.0)

    for input_value, weight in zip(inputs, weights):
        weighted_sum = weighted_sum + input_value * weight

    return weighted_sum + bias


x0 = Value(4.0)
x1 = Value(2.0)
w0 = Value(0.25)
w1 = Value(3.0)
bias = Value(-1.0)

prediction = predict_one_neuron(
    inputs=[x0, x1],
    weights=[w0, w1],
    bias=bias,
)

prediction.backward()

print("prediction:", prediction)
print("weight grads for prediction:", w0.grad, w1.grad)
print("bias grad for prediction:", bias.grad)

prediction: Value(data=6.0, grad=1.0)
weight grads for prediction: 4.0 2.0
bias grad for prediction: 1.0


## Reading the reusable helper output

The prediction is still `6.0`, so the function matches the hand-written formula. The gradients also match the earlier prediction-gradient rule: `w0.grad` is `4.0`, `w1.grad` is `2.0`, and `bias.grad` is `1.0`.

This check tells us the helper did not break the teaching graph. We can now reuse the same function in later cells when the notebook moves from prediction gradients to loss gradients and weight updates.

## Train on one example for several steps

The previous cell showed one update. This loop repeats the same training step ten times on the same example: make a prediction, compute the loss, call `.backward()`, and move each parameter using its gradient.

Before each backward pass, the parameter gradients are reset to `0.0`. The `Value` helper adds gradient contributions with `+=`, so resetting prevents old gradients from being reused by accident.

In [25]:
learning_rate = 0.01

x0 = Value(4.0)
x1 = Value(2.0)

w0 = Value(0.25)
w1 = Value(3.0)
bias = Value(-1.0)

target = Value(10.0)

for step in range(10):
    w0.grad = 0.0
    w1.grad = 0.0
    bias.grad = 0.0

    prediction = predict_one_neuron(
        inputs=[x0, x1],
        weights=[w0, w1],
        bias=bias,
    )

    error = prediction - target
    loss = error**2

    loss.backward()

    w0.data = w0.data - learning_rate * w0.grad
    w1.data = w1.data - learning_rate * w1.grad
    bias.data = bias.data - learning_rate * bias.grad

    print(step, prediction.data, loss.data)

0 6.0 16.0
1 7.6800000000000015 5.382399999999993
2 8.6544 1.810639359999998
3 9.219552 0.6090990807039997
4 9.54734016 0.20490093074882634
5 9.7374572928 0.06892867310390478
6 9.847725229824 0.023187605632153517
7 9.91168063329792 0.0078003105346565686
8 9.948774767312793 0.0026240244638584915
9 9.97028936504142 0.0008827218296419733


## Reading the one-example training loop

The printed prediction moves from `6.0` toward the target `10.0`. The printed loss gets smaller each step, which means the repeated gradient updates are improving this one example.

This is useful, but it is still only one row. A model should usually learn weights that work across several examples, so the next step is to add the losses from a tiny dataset.

## Compute loss across a tiny dataset

A dataset is a list of examples. Each example has input numbers and one target number, and the same weights are reused for every row.

This cell computes one loss per example, adds those losses into `total_loss`, and then prints the mean loss. The mean loss is just the total divided by the number of examples, so it gives one easy-to-read score for the whole tiny dataset.

In [26]:
training_examples = [
    ([4.0, 2.0], 10.0),
    ([2.0, 1.0], 5.0),
    ([1.0, 3.0], 7.0),
]

w0 = Value(0.25)
w1 = Value(3.0)
bias = Value(-1.0)

total_loss = Value(0.0)

for input_numbers, target_number in training_examples:
    x0 = Value(input_numbers[0])
    x1 = Value(input_numbers[1])
    target = Value(target_number)

    prediction = predict_one_neuron(
        inputs=[x0, x1],
        weights=[w0, w1],
        bias=bias,
    )

    error = prediction - target
    loss = error**2

    total_loss = total_loss + loss

mean_loss = Value(total_loss.data / len(training_examples))

print("total loss:", total_loss.data)
print("mean loss:", mean_loss.data)

total loss: 23.8125
mean loss: 7.9375


## Reading the tiny dataset loss output

The total loss is the sum of the three squared-error losses. The mean loss is that total divided by `3`, because there are three training examples.

This first dataset cell is for measuring, not training. It uses `mean_loss = Value(total_loss.data / len(training_examples))` only to display the average, so the mean-loss value is not connected to the gradient graph yet.

## Train on the tiny dataset: branches in simple words

Each training example makes its own prediction, error, and loss. You can think of each example as one route, or branch, from the shared weights to one loss value.

The same `w0`, `w1`, and `bias` are reused in every branch. When the code adds the example losses into `total_loss`, the graph connects all branches to one final value.

Calling `total_loss.backward()` starts from that final value and walks backward through every branch. Each branch adds its own gradient contribution into the same `w0.grad`, `w1.grad`, and `bias.grad`, which is also how PyTorch handles a summed batch loss.

In [31]:
learning_rate = 0.01

training_examples = [
    ([4.0, 2.0], 10.0),
    ([2.0, 1.0], 5.0),
    ([1.0, 3.0], 7.0),
]

w0 = Value(0.25)
w1 = Value(3.0)
bias = Value(-1.0)

for step in range(20):
    w0.grad = 0.0
    w1.grad = 0.0
    bias.grad = 0.0

    total_loss = Value(0.0)

    for input_numbers, target_number in training_examples:
        x0 = Value(input_numbers[0])
        x1 = Value(input_numbers[1])
        target = Value(target_number)

        prediction = predict_one_neuron(
            inputs=[x0, x1],
            weights=[w0, w1],
            bias=bias,
        )

        error = prediction - target
        loss = error**2
        total_loss = total_loss + loss

    total_loss.backward()

    w0.data = w0.data - learning_rate * w0.grad
    w1.data = w1.data - learning_rate * w1.grad
    bias.data = bias.data - learning_rate * bias.grad

    mean_loss = total_loss.data / len(training_examples)
    print(step, mean_loss)

0 7.9375
1 3.6623166666666656
2 2.775090456666669
3 2.3151043698306655
4 1.9578478241155552
5 1.6598986518314065
6 1.4090454220072208
7 1.1975805327615254
8 1.0192816650476149
9 0.8689334452229321
10 0.7421432340163093
11 0.6352090724873388
12 0.5450108414606407
13 0.46891878728960507
14 0.4047164515632275
15 0.35053571629659624
16 0.3048020578880854
17 0.2661884051310815
18 0.23357624901657267
19 0.20602286461437047


## Reading the dataset training output

The mean loss starts at `7.9375` and gets smaller over the training steps. That means the shared weights are changing in a direction that helps the dataset overall, not just one example.

The gradient for a shared parameter is the sum of the branch contributions. In plain words: each example gives an opinion about how `w0`, `w1`, and `bias` should move, and `total_loss.backward()` adds those opinions together before the update.